In [1]:
import requests
import os
import time
import zipfile
import pandas as pd
from minio import Minio
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql import types

In [2]:
client = Minio(
    "minio:9000",
    access_key="minioadmin",
    secret_key="minioadmin",
    secure=False
)
bucket = "crypto-data-lake"
if not client.bucket_exists(bucket):
    client.make_bucket(bucket)

In [3]:
spark = SparkSession.builder \
    .appName("LandingZone") \
    .config("spark.master", "spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.access.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key", "minioadmin") \
    .config("spark.hadoop.fs.s3a.endpoint", "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.path.style.access", "true") \
    .config("spark.jars", ",".join([
        "/opt/spark-extra-jars/hadoop-aws-3.3.4.jar",
        "/opt/spark-extra-jars/aws-java-sdk-bundle-1.12.262.jar"
    ])) \
    .getOrCreate()

25/09/22 04:16:31 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).


In [4]:
def download_file(url, file_name):
    if os.path.exists(file_name):
        print(f"{file_name} exits")
        return
    with requests.get(url, stream=True) as r:
        r.raise_for_status()
        with open(file_name, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        print(f"Downloaded {file_name} {(os.path.getsize(file_name) / (1024 * 1024)):.2f}MB completed")

In [5]:
def remove_file(file_name):
    if os.path.exists(file_name):
        os.remove(file_name)
        print(f"{file_name} removed")
    else:
        print(f"{file_name} not found")

In [6]:
def extract_file(extract_dir, zip_path):
    if not os.path.exists(extract_dir):
        os.makedirs(extract_dir)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(extract_dir)

In [7]:
extract_dir = "unzipped_data"

In [8]:
urls_month = [f"https://data.binance.vision/data/spot/monthly/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2022-{i:02d}.zip" for i in range(1, 3)]

In [9]:
urls = [f"https://data.binance.vision/data/spot/daily/aggTrades/BTCUSDT/BTCUSDT-aggTrades-2025-08-{i:02d}.zip" for i in range(1, 2)]

In [10]:
file_names = [u.split("/")[-1] for u in urls_month]

In [71]:
for url, file_name in zip(urls_month, file_names):
    start_t = time.time()
    download_file(url, file_name)
    # remove_file(file_name)
    end_t = time.time()
    print(f"Processed in {(end_t - start_t):.3f} seconds")

Downloaded BTCUSDT-aggTrades-2025-08-01.zip 18.64MB completed
Processed in 3.923 seconds


In [11]:
zip_path = "BTCUSDT-aggTrades-2025-08-01.zip"
csv_file = os.path.join(extract_dir, os.listdir(extract_dir)[0])
print(f"Extracted CSV: {csv_file}")

Extracted CSV: unzipped_data/BTCUSDT-aggTrades-2025-08-01.csv


In [12]:
columns = ["agg_trade_id","price","quantity","first_trade_id","last_trade_id","timestamp","is_buyer_maker","is_best_match"]
df_pandas = pd.read_csv("unzipped_data/BTCUSDT-aggTrades-2025-08-01.csv", header=None, names=columns)

In [13]:
df_pandas["datetime"] = pd.to_datetime(df_pandas["timestamp"], unit="us")

In [28]:
pd.set_option("display.float_format", "{:.10f}".format)

In [16]:
spark.createDataFrame(df_pandas).schema

StructType([StructField('agg_trade_id', LongType(), True), StructField('price', DoubleType(), True), StructField('quantity', DoubleType(), True), StructField('first_trade_id', LongType(), True), StructField('last_trade_id', LongType(), True), StructField('timestamp', LongType(), True), StructField('is_buyer_maker', BooleanType(), True), StructField('is_best_match', BooleanType(), True), StructField('datetime', TimestampType(), True)])

In [17]:
schema = types.StructType([
    types.StructField('agg_trade_id', types.LongType(), True), 
    types.StructField('price', types.DoubleType(), True), 
    types.StructField('quantity', types.DoubleType(), True), 
    types.StructField('first_trade_id', types.LongType(), True), 
    types.StructField('last_trade_id', types.LongType(), True), 
    types.StructField('timestamp', types.LongType(), True), 
    types.StructField('is_buyer_maker', types.BooleanType(), True), 
    types.StructField('is_best_match', types.BooleanType(), True)
])

In [18]:
df = spark.read \
    .option("header", "false") \
    .schema(schema) \
    .csv(csv_file)

In [21]:
df_pandas.dtypes

agg_trade_id               int64
price                    float64
quantity                 float64
first_trade_id             int64
last_trade_id              int64
timestamp                  int64
is_buyer_maker              bool
is_best_match               bool
datetime          datetime64[ns]
dtype: object

In [22]:
df.printSchema()

root
 |-- agg_trade_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- quantity: double (nullable = true)
 |-- first_trade_id: long (nullable = true)
 |-- last_trade_id: long (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- is_buyer_maker: boolean (nullable = true)
 |-- is_best_match: boolean (nullable = true)



In [23]:
df_pandas.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1314072 entries, 0 to 1314071
Data columns (total 9 columns):
 #   Column          Non-Null Count    Dtype         
---  ------          --------------    -----         
 0   agg_trade_id    1314072 non-null  int64         
 1   price           1314072 non-null  float64       
 2   quantity        1314072 non-null  float64       
 3   first_trade_id  1314072 non-null  int64         
 4   last_trade_id   1314072 non-null  int64         
 5   timestamp       1314072 non-null  int64         
 6   is_buyer_maker  1314072 non-null  bool          
 7   is_best_match   1314072 non-null  bool          
 8   datetime        1314072 non-null  datetime64[ns]
dtypes: bool(2), datetime64[ns](1), float64(2), int64(4)
memory usage: 72.7 MB


In [37]:
df.select([F.sum(F.col(c).isNull().cast("int")).alias(c) for c in df.columns]).show()

+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|agg_trade_id|price|quantity|first_trade_id|last_trade_id|timestamp|is_buyer_maker|is_best_match|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+
|           0|    0|       0|             0|            0|        0|             0|            0|
+------------+-----+--------+--------------+-------------+---------+--------------+-------------+



In [24]:
df_pandas.describe()

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,datetime
count,1314072,1314072,1314072,1314072,1314072,1314072,1314072
mean,3641151156,114773,0,5124889107,5124889109,1754049467280083,2025-08-01 11:57:47.280082944
min,3640494121,112723,0,5122977554,5122977554,1754006400328945,2025-08-01 00:00:00.328945
25%,3640822639,114223,0,5123834534,5123834534,1754028715030710,2025-08-01 06:11:55.030710528
50%,3641151156,114955,0,5124730672,5124730674,1754053884711544,2025-08-01 13:11:24.711544064
75%,3641479674,115431,0,5125972465,5125972466,1754071435964598,2025-08-01 18:03:55.964598272
max,3641808192,116052,27,5127138382,5127138382,1754092799895345,2025-08-01 23:59:59.895345
std,379340,815,0,1227424,1227425,25312150729,NaN


In [25]:
df.describe().show()

25/09/22 04:26:46 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|summary|     agg_trade_id|             price|            quantity|     first_trade_id|      last_trade_id|           timestamp|
+-------+-----------------+------------------+--------------------+-------------------+-------------------+--------------------+
|  count|          1314072|           1314072|             1314072|            1314072|            1314072|             1314072|
|   mean|   3.6411511565E9|114772.51094437456|0.018634520832961927|5.124889106730406E9|5.124889108896769E9|1.754049467280027...|
| stddev|379340.0558048148| 815.0145912458705| 0.13296616458618138| 1227424.3654440136| 1227424.8011467636|2.531215074523887...|
|    min|       3640494121|         112722.58|              1.0E-5|         5122977554|         5122977554|    1754006400328945|
|    max|       3641808192|          116052.0|             26.5822|         5127138382|         5

In [29]:
df_pandas.head(10)

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match,datetime
0,3640494121,115764.0700000000,0.2267700000,5122977554,5122977554,1754006400328945,True,True,2025-08-01 00:00:00.328945
1,3640494122,115764.0800000000,0.0014500000,5122977555,5122977555,1754006400345714,False,True,2025-08-01 00:00:00.345714
2,3640494123,115764.0800000000,0.0002100000,5122977556,5122977556,1754006400350235,False,True,2025-08-01 00:00:00.350235
3,3640494124,115764.0800000000,0.0004100000,5122977557,5122977557,1754006400492405,False,True,2025-08-01 00:00:00.492405
4,3640494125,115764.0800000000,0.0000800000,5122977558,5122977558,1754006400636161,False,True,2025-08-01 00:00:00.636161
5,3640494126,115764.0700000000,0.0044400000,5122977559,5122977559,1754006400669861,True,True,2025-08-01 00:00:00.669861
6,3640494127,115764.0700000000,0.1087700000,5122977560,5122977576,1754006400781656,True,True,2025-08-01 00:00:00.781656
7,3640494128,115764.0600000000,0.0001000000,5122977577,5122977577,1754006400781656,True,True,2025-08-01 00:00:00.781656
8,3640494129,115762.8900000000,0.0001500000,5122977578,5122977580,1754006400781656,True,True,2025-08-01 00:00:00.781656
9,3640494130,115762.8800000000,0.2852800000,5122977581,5122977584,1754006400781656,True,True,2025-08-01 00:00:00.781656


In [27]:
df.show(10, truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|3640494121  |115764.07|0.22677 |5122977554    |5122977554   |1754006400328945|true          |true         |
|3640494122  |115764.08|0.00145 |5122977555    |5122977555   |1754006400345714|false         |true         |
|3640494123  |115764.08|2.1E-4  |5122977556    |5122977556   |1754006400350235|false         |true         |
|3640494124  |115764.08|4.1E-4  |5122977557    |5122977557   |1754006400492405|false         |true         |
|3640494125  |115764.08|8.0E-5  |5122977558    |5122977558   |1754006400636161|false         |true         |
|3640494126  |115764.07|0.00444 |5122977559    |5122977559   |1754006400669861|true          |true         |
|3640494127  |11576

In [36]:
df.select(F.format_number("quantity", 10).alias("quantity")).show(10, truncate=False)

+------------+
|quantity    |
+------------+
|0.2267700000|
|0.0014500000|
|0.0002100000|
|0.0004100000|
|0.0000800000|
|0.0044400000|
|0.1087700000|
|0.0001000000|
|0.0001500000|
|0.2852800000|
+------------+
only showing top 10 rows



In [31]:
df_pandas.tail(10)

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match,datetime
1314062,3641808183,113297.9200000000,0.0601800000,5127138373,5127138373,1754092797316658,True,True,2025-08-01 23:59:57.316658
1314063,3641808184,113297.9300000000,0.0041200000,5127138374,5127138374,1754092797638613,False,True,2025-08-01 23:59:57.638613
1314064,3641808185,113297.9300000000,0.0000900000,5127138375,5127138375,1754092798129136,False,True,2025-08-01 23:59:58.129136
1314065,3641808186,113297.9200000000,0.0002700000,5127138376,5127138376,1754092798465431,True,True,2025-08-01 23:59:58.465431
1314066,3641808187,113297.9200000000,0.2772300000,5127138377,5127138377,1754092798510998,True,True,2025-08-01 23:59:58.510998
1314067,3641808188,113297.9200000000,0.0125700000,5127138378,5127138378,1754092798984525,True,True,2025-08-01 23:59:58.984525
1314068,3641808189,113297.9300000000,0.0071500000,5127138379,5127138379,1754092799147533,False,True,2025-08-01 23:59:59.147533
1314069,3641808190,113297.9300000000,0.0041200000,5127138380,5127138380,1754092799645164,False,True,2025-08-01 23:59:59.645164
1314070,3641808191,113297.9300000000,0.0004400000,5127138381,5127138381,1754092799878603,False,True,2025-08-01 23:59:59.878603
1314071,3641808192,113297.9300000000,0.0044100000,5127138382,5127138382,1754092799895345,False,True,2025-08-01 23:59:59.895345


In [32]:
df.orderBy("timestamp", ascending=False).show(10, truncate=False)

[Stage 4:=============================>                             (1 + 1) / 2]

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|3641808192  |113297.93|0.00441 |5127138382    |5127138382   |1754092799895345|false         |true         |
|3641808191  |113297.93|4.4E-4  |5127138381    |5127138381   |1754092799878603|false         |true         |
|3641808190  |113297.93|0.00412 |5127138380    |5127138380   |1754092799645164|false         |true         |
|3641808189  |113297.93|0.00715 |5127138379    |5127138379   |1754092799147533|false         |true         |
|3641808188  |113297.92|0.01257 |5127138378    |5127138378   |1754092798984525|true          |true         |
|3641808187  |113297.92|0.27723 |5127138377    |5127138377   |1754092798510998|true          |true         |
|3641808186  |11329

In [33]:
df_pandas.sample(10)

,agg_trade_id,price,quantity,first_trade_id,last_trade_id,timestamp,is_buyer_maker,is_best_match,datetime
861201,3641355322,114860.3400000000,0.0174100000,5125478463,5125478463,1754062075956780,False,True,2025-08-01 15:27:55.956780
435424,3640929545,114622.0900000000,0.0144700000,5124102764,5124102764,1754036476782537,False,True,2025-08-01 08:21:16.782537
170570,3640664691,115256.9600000000,0.0002800000,5123411309,5123411309,1754012372867815,True,True,2025-08-01 01:39:32.867815
1230198,3641724319,113227.2500000000,0.0014700000,5126856417,5126856417,1754087293332979,True,True,2025-08-01 22:28:13.332979
465717,3640959838,114476.0000000000,0.0020000000,5124164140,5124164140,1754037601340320,True,True,2025-08-01 08:40:01.340320
550565,3641044686,114894.3400000000,0.0120000000,5124387928,5124387928,1754045333703826,True,True,2025-08-01 10:48:53.703826
1307590,3641801711,113347.9900000000,0.0011600000,5127117967,5127117971,1754091755578446,False,True,2025-08-01 23:42:35.578446
764074,3641258195,115109.9800000000,0.0545500000,5125109576,5125109578,1754057502176974,True,True,2025-08-01 14:11:42.176974
117110,3640611231,115141.6400000000,0.0007400000,5123265662,5123265662,1754010412427099,False,True,2025-08-01 01:06:52.427099
1077356,3641571477,113403.0100000000,0.0045900000,5126322177,5126322177,1754076085069855,False,True,2025-08-01 19:21:25.069855


In [34]:
df.sample(0.001).show(10)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+
|  3640495955|115724.16|  4.3E-4|    5122981093|   5122981093|1754006540725541|         false|         true|
|  3640497481| 115780.0| 0.01097|    5122984382|   5122984397|1754006661715662|          true|         true|
|  3640498761|115788.01| 0.00152|    5122987403|   5122987403|1754006751959396|         false|         true|
|  3640499986| 115763.5| 0.00131|    5122990178|   5122990178|1754006879396399|         false|         true|
|  3640501078|115750.97|  3.2E-4|    5122992729|   5122992729|1754006981551096|          true|         true|
|  3640501535| 115699.1|  5.0E-5|    5122994219|   5122994219|1754007014215553|          true|         true|
|  3640502536|11555

In [49]:
df.withColumn("quantity_str", F.format_number(F.col("quantity"), 10).cast("string")).show(10, truncate=False)

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+------------+
|agg_trade_id|price    |quantity|first_trade_id|last_trade_id|timestamp       |is_buyer_maker|is_best_match|quantity_str|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+------------+
|3640494121  |115764.07|0.22677 |5122977554    |5122977554   |1754006400328945|true          |true         |0.2267700000|
|3640494122  |115764.08|0.00145 |5122977555    |5122977555   |1754006400345714|false         |true         |0.0014500000|
|3640494123  |115764.08|2.1E-4  |5122977556    |5122977556   |1754006400350235|false         |true         |0.0002100000|
|3640494124  |115764.08|4.1E-4  |5122977557    |5122977557   |1754006400492405|false         |true         |0.0004100000|
|3640494125  |115764.08|8.0E-5  |5122977558    |5122977558   |1754006400636161|false         |true         |0.0000800000|
|3640494126  |115764.07|

In [38]:
df.withColumn("datetime", F.from_unixtime(F.col("timestamp") / 1_000_000)).show()

+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-------------------+
|agg_trade_id|    price|quantity|first_trade_id|last_trade_id|       timestamp|is_buyer_maker|is_best_match|           datetime|
+------------+---------+--------+--------------+-------------+----------------+--------------+-------------+-------------------+
|  3640494121|115764.07| 0.22677|    5122977554|   5122977554|1754006400328945|          true|         true|2025-08-01 00:00:00|
|  3640494122|115764.08| 0.00145|    5122977555|   5122977555|1754006400345714|         false|         true|2025-08-01 00:00:00|
|  3640494123|115764.08|  2.1E-4|    5122977556|   5122977556|1754006400350235|         false|         true|2025-08-01 00:00:00|
|  3640494124|115764.08|  4.1E-4|    5122977557|   5122977557|1754006400492405|         false|         true|2025-08-01 00:00:00|
|  3640494125|115764.08|  8.0E-5|    5122977558|   5122977558|1754006400636161|         false|   

In [35]:
output_path = "parquet_data/aggTrades_2025_08_01"
df.write.mode("overwrite").parquet(output_path)
print(f"Parquet written to: {output_path}")

[Stage 5:>                                                          (0 + 2) / 2]

Parquet written to: parquet_data/aggTrades_2025_08_01


In [39]:
output_path = f"s3a://{bucket}/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01"
df.write.mode("overwrite").parquet(output_path)
print(f"Parquet written to: {output_path}")

25/09/21 14:56:46 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
                                                                                

Parquet written to: s3a://crypto-data-lake/landing_zone/spot/daily/aggTrades/BTCUSDT/2025_08_01


In [40]:
new_df = spark.read.parquet(output_path)

In [1]:
klines_columns = ["Open Time","Open Price","High Price","Low Price","Close Price","Volume","Close Time","Quote Asset Volume","Number of Trades","Taker Buy Base Asset Volume","Taker Buy Quote Asset Volume","Ignore"]